In [6]:
import io
import os
import boto3
import duckdb
import pandas as pd
from datasets import load_dataset

In [9]:
S3_ENDPOINT = os.environ.get("S3_ENDPOINT", "http://rustfs:9000")
BUCKET = "lakehouse"

s3 = boto3.client(
    "s3",
    endpoint_url=S3_ENDPOINT,
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
)
print("S3 client ready:", S3_ENDPOINT)

S3 client ready: http://rustfs:9000


In [10]:
print("Loading detection-datasets/coco val split...")
ds = load_dataset("detection-datasets/coco", split="val")
print(ds.features)
print(f"Total images: {len(ds)}")

Loading detection-datasets/coco val split...


Resolving data files:   0%|          | 0/40 [00:00<?, ?it/s]

{'image_id': Value('int64'), 'image': Image(mode=None, decode=True), 'width': Value('int64'), 'height': Value('int64'), 'objects': {'bbox_id': List(Value('int64')), 'category': List(ClassLabel(names=['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', '

In [11]:
rows = []

for i, sample in enumerate(ds):
    image_id = sample["image_id"]
    image = sample["image"]

    buf = io.BytesIO()
    image.save(buf, format="JPEG")
    buf.seek(0)
    key = f"assets/coco/images/{image_id}.jpg"
    s3.upload_fileobj(buf, BUCKET, key)
    image_uri = f"s3://{BUCKET}/{key}"

    objects = sample["objects"]
    for j in range(len(objects["bbox_id"])):
        rows.append({
            "image_uri": image_uri,
            "image_id": image_id,
            "width": sample["width"],
            "height": sample["height"],
            "bbox_id": objects["bbox_id"][j],
            "category": objects["category"][j],
            "bbox": str(objects["bbox"][j]),
            "area": objects["area"][j],
        })

    if (i + 1) % 100 == 0:
        print(f"{i + 1}/{len(ds)} images processed")

100/4952 images processed
200/4952 images processed
300/4952 images processed
400/4952 images processed
500/4952 images processed
600/4952 images processed
700/4952 images processed
800/4952 images processed
900/4952 images processed
1000/4952 images processed
1100/4952 images processed
1200/4952 images processed
1300/4952 images processed
1400/4952 images processed
1500/4952 images processed
1600/4952 images processed
1700/4952 images processed
1800/4952 images processed
1900/4952 images processed
2000/4952 images processed
2100/4952 images processed
2200/4952 images processed
2300/4952 images processed
2400/4952 images processed
2500/4952 images processed
2600/4952 images processed
2700/4952 images processed
2800/4952 images processed
2900/4952 images processed
3000/4952 images processed
3100/4952 images processed
3200/4952 images processed
3300/4952 images processed
3400/4952 images processed
3500/4952 images processed
3600/4952 images processed
3700/4952 images processed
3800/4952 

In [12]:
df = pd.DataFrame(rows)
print(f"Total annotation rows: {len(df)}")
df.head()

Total annotation rows: 36781


,image_uri,image_id,width,height,bbox_id,category,bbox,area
0,s3://lakehouse/assets/coco/images/139.jpg,139,640,426,26547,58,"[236.98, 142.51, 261.68, 212.01]",531.80710
1,s3://lakehouse/assets/coco/images/139.jpg,139,640,426,34646,62,"[7.03, 167.76, 156.35, 262.63]",13244.65770
2,s3://lakehouse/assets/coco/images/139.jpg,139,640,426,35802,62,"[557.21, 209.19, 638.5600000000001, 287.92]",5833.11795
3,s3://lakehouse/assets/coco/images/139.jpg,139,640,426,103487,56,"[358.98, 218.05, 414.98, 320.88]",2245.34355
4,s3://lakehouse/assets/coco/images/139.jpg,139,640,426,104368,56,"[290.69, 218.0, 352.52, 316.48]",1833.78400


In [13]:
con = duckdb.connect()
con.execute(open("/workspace/sql/00_attach.sql").read())
con.execute("CREATE TABLE raw.coco_annotations AS SELECT * FROM df")
print("Table created.")

CatalogException: Catalog Error: Table with name "coco_annotations" already exists!

In [8]:
print(con.sql("FROM ducklake_snapshots('lake')").df())
print(con.sql("SELECT COUNT(*) FROM raw.coco_annotations").df())
con.sql("SELECT image_uri, category, bbox FROM raw.coco_annotations LIMIT 5").df()
con.close()

   snapshot_id                    snapshot_time  schema_version  \
0            0 2026-06-25 19:47:58.592626+00:00               0   
1            1 2026-06-25 19:47:58.638004+00:00               1   
2            2 2026-06-25 19:47:58.647187+00:00               2   
3            3 2026-06-25 19:47:58.669349+00:00               3   
4            4 2026-06-25 22:18:44.460503+00:00               4   
5            5 2026-06-25 23:13:50.427815+00:00               5   
6            6 2026-06-25 23:13:50.502705+00:00               6   

                                             changes author commit_message  \
0                      {'schemas_created': ['main']}   None           None   
1                       {'schemas_created': ['raw']}   None           None   
2                    {'schemas_created': ['silver']}   None           None   
3                      {'schemas_created': ['gold']}   None           None   
4  {'tables_created': ['raw.coco_annotations'], '...   None           Non